# 7 Open Prediction Hyperparameter Tuning v1

## 7.0 Purpose

This notebook tunes the Stage 6 shortlisted open-prediction candidates using only the train-validation pool.

Shortlisted candidates:
1. Logistic Regression + C_expanded
2. Decision Tree + C_expanded
3. Random Forest + C_expanded

Primary tuning metric:
- ROC-AUC

Supporting metrics:
- PR-AUC / average precision
- Brier score
- Log loss
- Accuracy
- Balanced accuracy

Fixed rules:
- The final chronological test set is not used during tuning.
- Hyperparameters are selected using grouped cross-validation only.
- `mailing_id` is used as the grouping variable.
- Preprocessing remains inside sklearn Pipeline / ColumnTransformer.
- Train and validation scores are saved to diagnose overfitting and underfitting.
- If two settings perform similarly, the simpler or more stable setting is preferred.

## 7.1 Imports and paths

In [1]:
# imports
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedGroupKFold
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    accuracy_score,
    balanced_accuracy_score
)

In [13]:
# project paths
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "Data" / "Processed Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"

OPEN_DATA_PATH = DATA_DIR / "df_open_model_v1.parquet"

SHORTLIST_PATH = OUTPUT_DIR / "6_open_shortlist_v1.csv"

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

print("\nOpen data exists:", OPEN_DATA_PATH.exists())
print("Shortlist exists:", SHORTLIST_PATH.exists())

Project root: /Users/khanhhien/Documents/Thesis
Data dir: /Users/khanhhien/Documents/Thesis/Data/Processed Data
Output dir: /Users/khanhhien/Documents/Thesis/Outputs

Open data exists: True
Shortlist exists: True


In [14]:
# reproducibility
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

print("Random state =", RANDOM_STATE)

Random state = 42


In [15]:
NOTEBOOK_NAME = "7_open_hyperparameter_tuning_v1"

TARGET_NAME = "open"

PRIMARY_METRIC = "roc_auc"

print(f"Notebook: {NOTEBOOK_NAME}")
print(f"Target: {TARGET_NAME}")
print(f"Primary metric: {PRIMARY_METRIC}")

Notebook: 7_open_hyperparameter_tuning_v1
Target: open
Primary metric: roc_auc


## 7.2 Load Stage 6 shortlist

In [16]:
# Stage 6 shortlist (frozen)
OPEN_SHORTLIST = [
    "logistic_regression__C_expanded",
    "decision_tree__C_expanded",
    "random_forest__C_expanded"
]

print("Open shortlist:")
for model in OPEN_SHORTLIST:
    print("-", model)

Open shortlist:
- logistic_regression__C_expanded
- decision_tree__C_expanded
- random_forest__C_expanded


## 7.3 Load open modelling dataset

In [17]:
# load open modelling dataset
df_open = pd.read_parquet(OPEN_DATA_PATH)

print("Open dataset loaded.")
print("Shape:", df_open.shape)
print("Columns:", df_open.shape[1])

Open dataset loaded.
Shape: (1057544, 38)
Columns: 38


In [18]:
# basic target and campaign checks
TARGET_COL = "open"
GROUP_COL = "mailing_id"

required_cols = [TARGET_COL, GROUP_COL, "user_id"]

missing_required_cols = [col for col in required_cols if col not in df_open.columns]

print("Missing required columns:", missing_required_cols)

print("\nTarget distribution:")
print(df_open[TARGET_COL].value_counts(dropna=False))

print("\nTarget rate:")
print(df_open[TARGET_COL].mean())

print("\nUnique campaigns:", df_open[GROUP_COL].nunique())
print("Campaign ID range:", df_open[GROUP_COL].min(), "to", df_open[GROUP_COL].max())

assert len(missing_required_cols) == 0, "Some required columns are missing."
assert df_open[TARGET_COL].isna().sum() == 0, "Target contains missing values."

Missing required columns: []

Target distribution:
open
1.0    567034
0.0    490510
Name: count, dtype: int64

Target rate:
0.5361800549197008

Unique campaigns: 274
Campaign ID range: 653 to 1418


## 7.4 Recreate chronological campaign split

In [19]:
# chronological campaign holdout
FINAL_TEST_SIZE = 0.20
TRAINVAL_SIZE = 1 - FINAL_TEST_SIZE

campaign_ids = np.sort(df_open[GROUP_COL].dropna().unique())

n_campaigns = len(campaign_ids)
n_trainval_campaigns = int(np.floor(n_campaigns * TRAINVAL_SIZE))

trainval_campaigns = campaign_ids[:n_trainval_campaigns]
test_campaigns = campaign_ids[n_trainval_campaigns:]

trainval_df = df_open[df_open[GROUP_COL].isin(trainval_campaigns)].copy()
test_df = df_open[df_open[GROUP_COL].isin(test_campaigns)].copy()

print("Total campaigns:", n_campaigns)
print("Train-validation campaigns:", len(trainval_campaigns))
print("Final test campaigns:", len(test_campaigns))

print("\nTrain-validation shape:", trainval_df.shape)
print("Final test shape:", test_df.shape)

print("\nTrain-validation campaign range:", trainval_campaigns.min(), "to", trainval_campaigns.max())
print("Final test campaign range:", test_campaigns.min(), "to", test_campaigns.max())

Total campaigns: 274
Train-validation campaigns: 219
Final test campaigns: 55

Train-validation shape: (781826, 38)
Final test shape: (275718, 38)

Train-validation campaign range: 653 to 1252
Final test campaign range: 1257 to 1418


In [21]:
# split quality checks
campaign_overlap = set(trainval_campaigns).intersection(set(test_campaigns))

print("Campaign overlap:", len(campaign_overlap))

print("\nOpen rate:")
print("Train-validation:", trainval_df[TARGET_COL].mean())
print("Final test:", test_df[TARGET_COL].mean())

print("\nRows:")
print("Train-validation rows:", len(trainval_df))
print("Final test rows:", len(test_df))

assert len(campaign_overlap) == 0, "Campaign leakage: trainval and test share mailing_id."
assert len(trainval_df) + len(test_df) == len(df_open), "Split row counts do not sum to full dataset."

Campaign overlap: 0

Open rate:
Train-validation: 0.5732733370340716
Final test: 0.4309983388824814

Rows:
Train-validation rows: 781826
Final test rows: 275718


## 7.5 Feature set C definitions

In [23]:
open_feature_types_C = {
    "numeric": [
        "age_clean",
        "n_interests",
        "subject_length",
        "preheader_length",
        "prior_email_count",
        "prior_open_count",
        "prior_click_count",
        "historical_open_rate",
        "historical_click_rate",
    ],
    
    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket",
    ],
    
    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_missing",
        "subject_contains_number",
        "subject_contains_promotion",
        "subject_contains_exclamation",
        "preheader_missing",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "preheader_contains_exclamation",
        "had_prior_click",
        "is_first_email",
    ],
}

In [24]:
all_features_C = (
    open_feature_types_C["numeric"]
    + open_feature_types_C["categorical"]
    + open_feature_types_C["binary"]
)

print("Total features:", len(all_features_C))

missing_features = [
    col for col in all_features_C
    if col not in trainval_df.columns
]

print("Missing features:", missing_features)

Total features: 28
Missing features: []


## 7.6 Build preprocessing pipeline

In [25]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [26]:
# build preprocessing pipeline for Feature Set C
def build_preprocessor(feature_types):
    """
    Build a ColumnTransformer for numeric, categorical, and binary features.

    Important:
    - This preprocessor is not fitted here.
    - It will be fitted only inside each CV training fold through sklearn Pipeline.
    """

    numeric_features = feature_types["numeric"]
    categorical_features = feature_types["categorical"]
    binary_features = feature_types["binary"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    binary_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
            ("binary", binary_transformer, binary_features),
        ],
        remainder="drop"
    )

    return preprocessor

In [27]:
open_preprocessor_C = build_preprocessor(open_feature_types_C)

open_preprocessor_C

ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=0,
                                                                strategy='constant')),
                                                 ('scaler', StandardScaler())]),
                                 ['age_clean', 'n_interests', 'subject_length',
                                  'preheader_length', 'prior_email_count',
                                  'prior_open_count', 'prior_click_count',
                                  'historical_open_rate',
                                  'historical_click_rate']),
                                ('categorical',
                                 Pipeline(steps=...
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['interest_topic_match',
                                  'has_campaign_metadata', 'subject_missing',
                                  'subject_contains_number',
                                  'subject_contains_promotion',
                                  'subject_contains_exclamation',
                                  'preheader_missing',
                                  'preheader_contains_number',
                                  'preheader_contains_promotion',
                                  'preheader_contains_exclamation',
                                  'had_prior_click', 'is_first_email'])])

## 7.7 Create modelling matrices

In [28]:
# feature matrix, target, and grouping variable
all_features_C = (
    open_feature_types_C["numeric"]
    + open_feature_types_C["categorical"]
    + open_feature_types_C["binary"]
)

X_trainval = trainval_df[all_features_C].copy()

y_trainval = trainval_df[TARGET_COL].copy()

groups_trainval = trainval_df[GROUP_COL].copy()

print("X shape:", X_trainval.shape)
print("y shape:", y_trainval.shape)
print("groups shape:", groups_trainval.shape)

print("\nTarget rate:")
print(y_trainval.mean())

print("\nUnique campaigns:")
print(groups_trainval.nunique())

X shape: (781826, 28)
y shape: (781826,)
groups shape: (781826,)

Target rate:
0.5732733370340716

Unique campaigns:
219


In [29]:
# sanity checks
assert len(X_trainval) == len(y_trainval)
assert len(X_trainval) == len(groups_trainval)

assert X_trainval.index.equals(y_trainval.index)
assert X_trainval.index.equals(groups_trainval.index)

print("All sanity checks passed.")

All sanity checks passed.


## 7.8 StratifiedGroupKFold

In [30]:
# grouped cross-validation
cv_strategy = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print(cv_strategy)

StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)


In [31]:
# fold diagnostics
fold_summary = []

for fold_idx, (train_idx, val_idx) in enumerate(
    cv_strategy.split(
        X_trainval,
        y_trainval,
        groups=groups_trainval
    ),
    start=1
):

    y_train_fold = y_trainval.iloc[train_idx]
    y_val_fold = y_trainval.iloc[val_idx]

    groups_train_fold = set(groups_trainval.iloc[train_idx])
    groups_val_fold = set(groups_trainval.iloc[val_idx])

    overlap = groups_train_fold.intersection(groups_val_fold)

    fold_summary.append({
        "fold": fold_idx,
        "train_rows": len(train_idx),
        "val_rows": len(val_idx),
        "train_open_rate": y_train_fold.mean(),
        "val_open_rate": y_val_fold.mean(),
        "group_overlap": len(overlap)
    })

fold_summary_df = pd.DataFrame(fold_summary)

fold_summary_df

,fold,train_rows,val_rows,train_open_rate,val_open_rate,group_overlap
0,1,622701,159125,0.561046,0.621122,0
1,2,640613,141213,0.575329,0.563949,0
2,3,598719,183107,0.572579,0.575543,0
3,4,616035,165791,0.570283,0.584386,0
4,5,649236,132590,0.586451,0.508749,0


In [32]:
print(
    fold_summary_df[
        ["fold",
         "train_open_rate",
         "val_open_rate",
         "group_overlap"]
    ]
)

   fold  train_open_rate  val_open_rate  group_overlap
0     1         0.561046       0.621122              0
1     2         0.575329       0.563949              0
2     3         0.572579       0.575543              0
3     4         0.570283       0.584386              0
4     5         0.586451       0.508749              0


## 7.9 Define tuning metrics

In [34]:
scoring_open = {
    "ROC_AUC": "roc_auc",
    "PR_AUC": "average_precision",
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "neg_brier": "neg_brier_score",
    "neg_log_loss": "neg_log_loss",
}

open_refit_metric = "ROC_AUC"

print("Primary scoring metric:", open_refit_metric)
print("All scoring metrics:")
for metric in scoring_open:
    print("-", metric)

Primary scoring metric: ROC_AUC
All scoring metrics:
- ROC_AUC
- PR_AUC
- accuracy
- balanced_accuracy
- neg_brier
- neg_log_loss


In [35]:
# output file names

OPEN_TUNING_RESULTS_PATH = OUTPUT_DIR / "7_open_tuning_results_v1.csv"
OPEN_BEST_PARAMS_PATH = OUTPUT_DIR / "7_open_best_params_v1.json"
OPEN_TUNED_SUMMARY_PATH = OUTPUT_DIR / "7_open_tuned_model_summary_v1.csv"
OPEN_TUNING_NOTES_PATH = OUTPUT_DIR / "7_open_tuning_notes_v1.md"

print("Tuning results path:", OPEN_TUNING_RESULTS_PATH)
print("Best params path:", OPEN_BEST_PARAMS_PATH)
print("Summary path:", OPEN_TUNED_SUMMARY_PATH)
print("Notes path:", OPEN_TUNING_NOTES_PATH)

Tuning results path: /Users/khanhhien/Documents/Thesis/Outputs/7_open_tuning_results_v1.csv
Best params path: /Users/khanhhien/Documents/Thesis/Outputs/7_open_best_params_v1.json
Summary path: /Users/khanhhien/Documents/Thesis/Outputs/7_open_tuned_model_summary_v1.csv
Notes path: /Users/khanhhien/Documents/Thesis/Outputs/7_open_tuning_notes_v1.md


## 7.10 Helper functions to extract GridSearchCV results

In [36]:
# GridSearchCV result extraction
def extract_grid_results(
    grid_search,
    candidate_name,
    primary_metric
):
    """
    Convert GridSearchCV cv_results_ into a clean dataframe.
    """

    results = pd.DataFrame(grid_search.cv_results_).copy()

    train_col = f"mean_train_{primary_metric}"
    val_col = f"mean_test_{primary_metric}"

    results["candidate"] = candidate_name

    results["train_val_gap"] = (
        results[train_col]
        - results[val_col]
    )

    return results

In [37]:
# best parameter extraction
def extract_best_model_summary(
    grid_search,
    candidate_name,
    primary_metric
):
    """
    Extract summary of the best GridSearchCV solution.
    """

    best_idx = grid_search.best_index_

    cv_results = pd.DataFrame(grid_search.cv_results_)

    row = cv_results.iloc[best_idx]

    summary = {
        "candidate": candidate_name,
        "best_params": grid_search.best_params_,
        "best_score": grid_search.best_score_,
        "train_score": row[f"mean_train_{primary_metric}"],
        "validation_score": row[f"mean_test_{primary_metric}"],
        "validation_std": row[f"std_test_{primary_metric}"],
        "train_val_gap":
            row[f"mean_train_{primary_metric}"]
            - row[f"mean_test_{primary_metric}"]
    }

    return summary

In [39]:
# Grid Search configuration

N_JOBS = 1

VERBOSE = 3

RETURN_TRAIN_SCORE = True

print("n_jobs =", N_JOBS)
print("verbose =", VERBOSE)
print("return_train_score =", RETURN_TRAIN_SCORE)

n_jobs = 1
verbose = 3
return_train_score = True


## 7.11 Define candidate pipelines

In [41]:
open_candidate_pipelines = {
    "logistic_regression__C_expanded": Pipeline([
        ("preprocessor", open_preprocessor_C),
        ("model", LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ]),

    "decision_tree__C_expanded": Pipeline([
        ("preprocessor", open_preprocessor_C),
        ("model", DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ))
    ]),

    "random_forest__C_expanded": Pipeline([
        ("preprocessor", open_preprocessor_C),
        ("model", RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS
        ))
    ]),
}

list(open_candidate_pipelines.keys())

['logistic_regression__C_expanded',
 'decision_tree__C_expanded',
 'random_forest__C_expanded']

## 7.12 Define hyperparameter grids

In [43]:
open_param_grids = {
    "logistic_regression__C_expanded": {
        "model__C": [0.001, 0.01, 0.1, 1, 10],
        "model__penalty": ["l2"],
        "model__class_weight": [None],
    },

    "decision_tree__C_expanded": {
        "model__max_depth": [3, 5, 7, 10],
        "model__min_samples_leaf": [250, 500, 1000],
    },

    "random_forest__C_expanded": {
        "model__n_estimators": [200],
        "model__max_depth": [10, 20, 30],
        "model__min_samples_leaf": [50, 100, 250],
        "model__max_features": ["sqrt", 0.5],
        "model__class_weight": [None],
    },
}

In [44]:
# check grid sizes
from sklearn.model_selection import ParameterGrid

for candidate, grid in open_param_grids.items():
    n_combinations = len(list(ParameterGrid(grid)))
    n_total_fits = n_combinations * cv_strategy.get_n_splits()

    print(candidate)
    print("  combinations:", n_combinations)
    print("  total CV fits:", n_total_fits)

logistic_regression__C_expanded
  combinations: 5
  total CV fits: 25
decision_tree__C_expanded
  combinations: 12
  total CV fits: 60
random_forest__C_expanded
  combinations: 18
  total CV fits: 90


## 7.13 Candidate registry

In [45]:
OPEN_PRIMARY_METRIC = "ROC_AUC"

OPEN_CANDIDATES = [
    "logistic_regression__C_expanded",
    "decision_tree__C_expanded",
    "random_forest__C_expanded"
]

print("Primary metric:", OPEN_PRIMARY_METRIC)

for candidate in OPEN_CANDIDATES:
    print("-", candidate)

Primary metric: ROC_AUC
- logistic_regression__C_expanded
- decision_tree__C_expanded
- random_forest__C_expanded


## 7.14 Tune Logistic Regression C

In [46]:
# Logistic Regression GridSearchCV
open_logistic_search = GridSearchCV(
    estimator=open_candidate_pipelines[
        "logistic_regression__C_expanded"
    ],

    param_grid=open_param_grids[
        "logistic_regression__C_expanded"
    ],

    scoring=scoring_open,

    refit=OPEN_PRIMARY_METRIC,

    cv=cv_strategy,

    n_jobs=N_JOBS,

    verbose=VERBOSE,

    return_train_score=RETURN_TRAIN_SCORE
)

open_logistic_search

GridSearchCV(cv=StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(fill_value=0,
                                                                                                        strategy='constant')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['age_clean',
                                                                          'n_interests',
                                                                          'subject_length',
                                                                          'preheader_length',
                                                                          'prior_email_count...
                                                           random_state=42))]),
             n_jobs=1,
             param_grid={'model__C': [0.001, 0.01, 0.1, 1, 10],
                         'model__class_weight': [None],
                         'model__penalty': ['l2']},
             refit='ROC_AUC', return_train_score=True,
             scoring={'PR_AUC': 'average_precision', 'ROC_AUC': 'roc_auc',
                      'accuracy': 'accuracy',
                      'balanced_accuracy': 'balanced_accuracy',
                      'neg_brier': 'neg_brier_score',
                      'neg_log_loss': 'neg_log_loss'},
             verbose=3)

In [47]:
print(open_logistic_search.refit)
print(open_logistic_search.scoring.keys())

ROC_AUC
dict_keys(['ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'neg_brier', 'neg_log_loss'])


In [ ]:
# fit Logistic Regression Grid Search
logistic_start_time = time.time()

open_logistic_search.fit(
    X_trainval,
    y_trainval,
    groups=groups_trainval
)

logistic_runtime = time.time() - logistic_start_time

print(f"Runtime: {logistic_runtime:.2f} seconds")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
[CV 1/5] END model__C=0.001, model__class_weight=None, model__penalty=l2; PR_AUC: (train=0.871, test=0.880) ROC_AUC: (train=0.852, test=0.838) accuracy: (train=0.773, test=0.773) balanced_accuracy: (train=0.766, test=0.752) neg_brier: (train=-0.155, test=-0.156) neg_log_loss: (train=-0.475, test=-0.479) total time=   4.9s
[CV 2/5] END model__C=0.001, model__class_weight=None, model__penalty=l2; PR_AUC: (train=0.879, test=0.852) ROC_AUC: (train=0.855, test=0.822) accuracy: (train=0.781, test=0.737) balanced_accuracy: (train=0.772, test=0.723) neg_brier: (train=-0.152, test=-0.171) neg_log_loss: (train=-0.467, test=-0.513) total time=   4.9s
[CV 3/5] END model__C=0.001, model__class_weight=None, model__penalty=l2; PR_AUC: (train=0.874, test=0.862) ROC_AUC: (train=0.852, test=0.841) accuracy: (train=0.773, test=0.770) balanced_accuracy: (train=0.764, test=0.767) neg_brier: (train=-0.154, test=-0.160) neg_log_loss: (train=-0.473, 

In [49]:
print("Best parameters:")
print(open_logistic_search.best_params_)

print("\nBest ROC-AUC:")
print(open_logistic_search.best_score_)

Best parameters:
{'model__C': 0.001, 'model__class_weight': None, 'model__penalty': 'l2'}

Best ROC-AUC:
0.830109893863732


In [50]:
# extract results
open_logistic_results = extract_grid_results(
    grid_search=open_logistic_search,
    candidate_name="logistic_regression__C_expanded",
    primary_metric=OPEN_PRIMARY_METRIC
)

open_logistic_summary = extract_best_model_summary(
    grid_search=open_logistic_search,
    candidate_name="logistic_regression__C_expanded",
    primary_metric=OPEN_PRIMARY_METRIC
)

open_logistic_summary

{'candidate': 'logistic_regression__C_expanded',
 'best_params': {'model__C': 0.001,
  'model__class_weight': None,
  'model__penalty': 'l2'},
 'best_score': 0.830109893863732,
 'train_score': 0.8519575166056048,
 'validation_score': 0.830109893863732,
 'validation_std': 0.01406335777848136,
 'train_val_gap': 0.021847622741872796}

In [51]:
# show key columns only
logistic_display_cols = [
    "candidate",
    "param_model__C",
    "mean_train_ROC_AUC",
    "mean_test_ROC_AUC",
    "std_test_ROC_AUC",
    "train_val_gap",
    "mean_test_PR_AUC",
    "mean_test_neg_brier",
    "mean_test_neg_log_loss",
    "rank_test_ROC_AUC"
]

open_logistic_results[logistic_display_cols].sort_values("rank_test_ROC_AUC")

,candidate,param_model__C,mean_train_ROC_AUC,mean_test_ROC_AUC,std_test_ROC_AUC,train_val_gap,mean_test_PR_AUC,mean_test_neg_brier,mean_test_neg_log_loss,rank_test_ROC_AUC
0,logistic_regression__C_expanded,0.001,0.851958,0.830110,0.014063,0.021848,0.845041,-0.165171,-0.504212,1
1,logistic_regression__C_expanded,0.010,0.853274,0.828542,0.015209,0.024732,0.845461,-0.167103,-0.508568,2
2,logistic_regression__C_expanded,0.100,0.853357,0.822371,0.023499,0.030986,0.837568,-0.170431,-0.518778,3
3,logistic_regression__C_expanded,1.000,0.853353,0.821929,0.024048,0.031424,0.836937,-0.170663,-0.519516,4
4,logistic_regression__C_expanded,10.000,0.853353,0.819988,0.027259,0.033365,0.832892,-0.171505,-0.522513,5


In [52]:
# save Logistic Regression tuning
logistic_results_path = OUTPUT_DIR / "7_open_logistic_tuning_results_v1.csv"
logistic_summary_path = OUTPUT_DIR / "7_open_logistic_tuning_summary_v1.json"

open_logistic_results.to_csv(logistic_results_path, index=False)

with open(logistic_summary_path, "w") as f:
    json.dump(open_logistic_summary, f, indent=4, default=str)

print("Saved logistic results to:", logistic_results_path)
print("Saved logistic summary to:", logistic_summary_path)

Saved logistic results to: /Users/khanhhien/Documents/Thesis/Outputs/7_open_logistic_tuning_results_v1.csv
Saved logistic summary to: /Users/khanhhien/Documents/Thesis/Outputs/7_open_logistic_tuning_summary_v1.json


Logistic Regression benefited from strong regularisation. Validation ROC-AUC decreased consistently as C increased, while train-validation gaps and fold variability increased. The best-performing configuration was C=0.001, which achieved the highest validation ROC-AUC, the smallest train-validation gap, and the most stable cross-validation performance. This suggests that stronger regularisation improves generalisation for the expanded feature set.

## 7.14 Tune Decision Tree C

In [53]:
# Decision Tree GridSearchCV
open_tree_search = GridSearchCV(
    estimator=open_candidate_pipelines[
        "decision_tree__C_expanded"
    ],
    param_grid=open_param_grids[
        "decision_tree__C_expanded"
    ],
    scoring=scoring_open,
    refit=OPEN_PRIMARY_METRIC,
    cv=cv_strategy,
    n_jobs=N_JOBS,
    verbose=VERBOSE,
    return_train_score=RETURN_TRAIN_SCORE
)

print("Refit metric:", open_tree_search.refit)
print("Scoring metrics:", open_tree_search.scoring.keys())

Refit metric: ROC_AUC
Scoring metrics: dict_keys(['ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'neg_brier', 'neg_log_loss'])


In [54]:
# fit Decision Tree Grid Search
tree_start_time = time.time()

open_tree_search.fit(
    X_trainval,
    y_trainval,
    groups=groups_trainval
)

tree_runtime = time.time() - tree_start_time

print(f"Runtime: {tree_runtime:.2f} seconds")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
[CV 1/5] END model__max_depth=3, model__min_samples_leaf=250; PR_AUC: (train=0.842, test=0.854) ROC_AUC: (train=0.845, test=0.831) accuracy: (train=0.774, test=0.779) balanced_accuracy: (train=0.764, test=0.757) neg_brier: (train=-0.155, test=-0.157) neg_log_loss: (train=-0.477, test=-0.482) total time=   2.9s
[CV 2/5] END model__max_depth=3, model__min_samples_leaf=250; PR_AUC: (train=0.847, test=0.823) ROC_AUC: (train=0.843, test=0.812) accuracy: (train=0.777, test=0.728) balanced_accuracy: (train=0.768, test=0.713) neg_brier: (train=-0.155, test=-0.172) neg_log_loss: (train=-0.478, test=-0.517) total time=   2.7s
[CV 3/5] END model__max_depth=3, model__min_samples_leaf=250; PR_AUC: (train=0.847, test=0.840) ROC_AUC: (train=0.845, test=0.838) accuracy: (train=0.772, test=0.783) balanced_accuracy: (train=0.759, test=0.776) neg_brier: (train=-0.155, test=-0.159) neg_log_loss: (train=-0.474, test=-0.511) total time=   2.8s
[CV

In [55]:
print("Best parameters:")
print(open_tree_search.best_params_)

print("\nBest ROC-AUC:")
print(open_tree_search.best_score_)

Best parameters:
{'model__max_depth': 7, 'model__min_samples_leaf': 1000}

Best ROC-AUC:
0.828697467639641


In [56]:
# extract Decision Tree tuning results
open_tree_results = extract_grid_results(
    grid_search=open_tree_search,
    candidate_name="decision_tree__C_expanded",
    primary_metric=OPEN_PRIMARY_METRIC
)

open_tree_summary = extract_best_model_summary(
    grid_search=open_tree_search,
    candidate_name="decision_tree__C_expanded",
    primary_metric=OPEN_PRIMARY_METRIC
)

open_tree_summary

{'candidate': 'decision_tree__C_expanded',
 'best_params': {'model__max_depth': 7, 'model__min_samples_leaf': 1000},
 'best_score': 0.828697467639641,
 'train_score': 0.8640854633198701,
 'validation_score': 0.828697467639641,
 'validation_std': 0.029220856097120176,
 'train_val_gap': 0.0353879956802291}

In [57]:
tree_display_cols = [
    "candidate",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "mean_train_ROC_AUC",
    "mean_test_ROC_AUC",
    "std_test_ROC_AUC",
    "train_val_gap",
    "mean_test_PR_AUC",
    "mean_test_neg_brier",
    "mean_test_neg_log_loss",
    "rank_test_ROC_AUC"
]

open_tree_results[tree_display_cols].sort_values("rank_test_ROC_AUC")

,candidate,param_model__max_depth,param_model__min_samples_leaf,mean_train_ROC_AUC,mean_test_ROC_AUC,std_test_ROC_AUC,train_val_gap,mean_test_PR_AUC,mean_test_neg_brier,mean_test_neg_log_loss,rank_test_ROC_AUC
8,decision_tree__C_expanded,7,1000,0.864085,0.828697,0.029221,0.035388,0.839603,-0.165934,-0.515456,1
11,decision_tree__C_expanded,10,1000,0.868608,0.828335,0.029145,0.040272,0.843659,-0.166817,-0.525167,2
0,decision_tree__C_expanded,3,250,0.842427,0.828011,0.016053,0.014416,0.826027,-0.163646,-0.504353,3
1,decision_tree__C_expanded,3,500,0.842427,0.828011,0.016053,0.014416,0.826027,-0.163646,-0.504353,3
2,decision_tree__C_expanded,3,1000,0.842427,0.828011,0.016053,0.014416,0.826027,-0.163646,-0.504353,3
5,decision_tree__C_expanded,5,1000,0.857722,0.827661,0.029523,0.030061,0.834224,-0.165614,-0.512128,6
4,decision_tree__C_expanded,5,500,0.857864,0.826390,0.032687,0.031475,0.832213,-0.166369,-0.514009,7
3,decision_tree__C_expanded,5,250,0.857988,0.826192,0.032661,0.031796,0.832130,-0.166490,-0.514466,8
6,decision_tree__C_expanded,7,250,0.865117,0.824618,0.036194,0.040499,0.835287,-0.168336,-0.526631,9
7,decision_tree__C_expanded,7,500,0.864763,0.823549,0.036690,0.041214,0.835197,-0.169087,-0.525445,10


In [58]:
# save Decision Tree tuning
tree_results_path = OUTPUT_DIR / "7_open_decision_tree_tuning_results_v1.csv"
tree_summary_path = OUTPUT_DIR / "7_open_decision_tree_tuning_summary_v1.json"

open_tree_results.to_csv(tree_results_path, index=False)

with open(tree_summary_path, "w") as f:
    json.dump(open_tree_summary, f, indent=4, default=str)

print("Saved tree results to:", tree_results_path)
print("Saved tree summary to:", tree_summary_path)

Saved tree results to: /Users/khanhhien/Documents/Thesis/Outputs/7_open_decision_tree_tuning_results_v1.csv
Saved tree summary to: /Users/khanhhien/Documents/Thesis/Outputs/7_open_decision_tree_tuning_summary_v1.json


Provisional preferred tree may be depth=3 because it is almost equal in ROC-AUC but simpler, more stable, and has a much smaller train-validation gap.

## 7.16 Tune Random Forest C

In [59]:
# random Forest GridSearchCV
open_rf_search = GridSearchCV(
    estimator=open_candidate_pipelines[
        "random_forest__C_expanded"
    ],
    param_grid=open_param_grids[
        "random_forest__C_expanded"
    ],
    scoring=scoring_open,
    refit=OPEN_PRIMARY_METRIC,
    cv=cv_strategy,
    n_jobs=N_JOBS,
    verbose=VERBOSE,
    return_train_score=RETURN_TRAIN_SCORE
)

print("Refit metric:", open_rf_search.refit)
print("Scoring metrics:", open_rf_search.scoring.keys())

Refit metric: ROC_AUC
Scoring metrics: dict_keys(['ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'neg_brier', 'neg_log_loss'])


In [60]:
# fit Random Forest Grid Search
rf_start_time = time.time()

open_rf_search.fit(
    X_trainval,
    y_trainval,
    groups=groups_trainval
)

rf_runtime = time.time() - rf_start_time

print(f"Runtime: {rf_runtime:.2f} seconds")

Fitting 5 folds for each of 18 candidates, totalling 90 fits
[CV 1/5] END model__class_weight=None, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=50, model__n_estimators=200; PR_AUC: (train=0.893, test=0.884) ROC_AUC: (train=0.875, test=0.846) accuracy: (train=0.793, test=0.780) balanced_accuracy: (train=0.789, test=0.762) neg_brier: (train=-0.143, test=-0.152) neg_log_loss: (train=-0.444, test=-0.471) total time= 1.0min
[CV 2/5] END model__class_weight=None, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=50, model__n_estimators=200; PR_AUC: (train=0.900, test=0.836) ROC_AUC: (train=0.877, test=0.798) accuracy: (train=0.796, test=0.721) balanced_accuracy: (train=0.789, test=0.701) neg_brier: (train=-0.141, test=-0.186) neg_log_loss: (train=-0.438, test=-0.550) total time= 1.1min
[CV 3/5] END model__class_weight=None, model__max_depth=10, model__max_features=sqrt, model__min_samples_leaf=50, model__n_estimators=200; PR_AUC: (train=0.896, 

In [61]:
print("Best parameters:")
print(open_rf_search.best_params_)

print("\nBest ROC-AUC:")
print(open_rf_search.best_score_)

Best parameters:
{'model__class_weight': None, 'model__max_depth': 30, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 250, 'model__n_estimators': 200}

Best ROC-AUC:
0.8372427378582342


In [62]:
open_rf_results = extract_grid_results(
    grid_search=open_rf_search,
    candidate_name="random_forest__C_expanded",
    primary_metric=OPEN_PRIMARY_METRIC
)

open_rf_summary = extract_best_model_summary(
    grid_search=open_rf_search,
    candidate_name="random_forest__C_expanded",
    primary_metric=OPEN_PRIMARY_METRIC
)

In [63]:
rf_display_cols = [
    "candidate",
    "param_model__n_estimators",
    "param_model__max_depth",
    "param_model__min_samples_leaf",
    "param_model__max_features",
    "mean_train_ROC_AUC",
    "mean_test_ROC_AUC",
    "std_test_ROC_AUC",
    "train_val_gap",
    "mean_test_PR_AUC",
    "rank_test_ROC_AUC"
]

open_rf_results[rf_display_cols].sort_values(
    "rank_test_ROC_AUC"
)

,candidate,param_model__n_estimators,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_train_ROC_AUC,mean_test_ROC_AUC,std_test_ROC_AUC,train_val_gap,mean_test_PR_AUC,rank_test_ROC_AUC
14,random_forest__C_expanded,200,30,250,sqrt,0.874846,0.837243,0.023368,0.037603,0.859746,1
12,random_forest__C_expanded,200,30,50,sqrt,0.885942,0.837104,0.027301,0.048838,0.860623,2
6,random_forest__C_expanded,200,20,50,sqrt,0.884992,0.837094,0.026881,0.047899,0.859637,3
13,random_forest__C_expanded,200,30,100,sqrt,0.880544,0.837052,0.025495,0.043492,0.859657,4
2,random_forest__C_expanded,200,10,250,sqrt,0.869760,0.836972,0.022037,0.032788,0.860264,5
1,random_forest__C_expanded,200,10,100,sqrt,0.871395,0.836935,0.023147,0.034460,0.860474,6
8,random_forest__C_expanded,200,20,250,sqrt,0.874815,0.836824,0.023456,0.037991,0.859421,7
7,random_forest__C_expanded,200,20,100,sqrt,0.880229,0.836676,0.025278,0.043554,0.859313,8
0,random_forest__C_expanded,200,10,50,sqrt,0.872240,0.835764,0.024523,0.036477,0.856690,9
9,random_forest__C_expanded,200,20,50,0.5,0.895552,0.832453,0.032250,0.063098,0.853058,10


In [64]:
# save Random Forest tuning backup
rf_results_path = OUTPUT_DIR / "7_open_random_forest_tuning_results_v1.csv"
rf_summary_path = OUTPUT_DIR / "7_open_random_forest_tuning_summary_v1.json"

open_rf_results.to_csv(
    rf_results_path,
    index=False
)

with open(rf_summary_path, "w") as f:
    json.dump(
        open_rf_summary,
        f,
        indent=4,
        default=str
    )

print("Saved RF results to:", rf_results_path)
print("Saved RF summary to:", rf_summary_path)

Saved RF results to: /Users/khanhhien/Documents/Thesis/Outputs/7_open_random_forest_tuning_results_v1.csv
Saved RF summary to: /Users/khanhhien/Documents/Thesis/Outputs/7_open_random_forest_tuning_summary_v1.json


GridSearchCV selected depth=30, leaf=250, sqrt by mean ROC-AUC, but depth=10, leaf=250, sqrt achieved almost identical ROC-AUC with a smaller train-validation gap and slightly lower fold variability.

## 7.17 Open tuned candidate comparison

This section compares the tuned open-prediction candidates using cross-validation results only.  
The final chronological test set is still not used.

The purpose is to decide which tuned configurations should be carried forward to the refit/final evaluation stage.

In [65]:
# combine tuned model summaries
open_model_summaries = [
    open_logistic_summary,
    open_tree_summary,
    open_rf_summary
]

open_tuned_summary_df = pd.DataFrame(
    open_model_summaries
)

open_tuned_summary_df

,candidate,best_params,best_score,train_score,validation_score,validation_std,train_val_gap
0,logistic_regression__C_expanded,"{'model__C': 0.001, 'model__class_weight': Non...",0.830110,0.851958,0.830110,0.014063,0.021848
1,decision_tree__C_expanded,"{'model__max_depth': 7, 'model__min_samples_le...",0.828697,0.864085,0.828697,0.029221,0.035388
2,random_forest__C_expanded,"{'model__class_weight': None, 'model__max_dept...",0.837243,0.874846,0.837243,0.023368,0.037603


In [67]:
# open tuned summary table
open_summary_display = open_tuned_summary_df.copy()

open_summary_display["conservative_score"] = (
    open_summary_display["validation_score"]
    - open_summary_display["validation_std"]
)

open_summary_display = open_summary_display[
    [
        "candidate",
        "best_params",
        "train_score",
        "validation_score",
        "validation_std",
        "conservative_score",
        "train_val_gap"
    ]
].sort_values("validation_score", ascending=False)

open_summary_display

,candidate,best_params,train_score,validation_score,validation_std,conservative_score,train_val_gap
2,random_forest__C_expanded,"{'model__class_weight': None, 'model__max_dept...",0.874846,0.837243,0.023368,0.813874,0.037603
0,logistic_regression__C_expanded,"{'model__C': 0.001, 'model__class_weight': Non...",0.851958,0.830110,0.014063,0.816047,0.021848
1,decision_tree__C_expanded,"{'model__max_depth': 7, 'model__min_samples_le...",0.864085,0.828697,0.029221,0.799477,0.035388


In [ ]:
# near-best decision tree alternatives
tree_near_best = open_tree_results[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "mean_train_ROC_AUC",
        "mean_test_ROC_AUC",
        "std_test_ROC_AUC",
        "train_val_gap",
        "mean_test_PR_AUC",
        "rank_test_ROC_AUC"
    ]
].sort_values("rank_test_ROC_AUC")

tree_near_best.head(6)

,param_model__max_depth,param_model__min_samples_leaf,mean_train_ROC_AUC,mean_test_ROC_AUC,std_test_ROC_AUC,train_val_gap,mean_test_PR_AUC,rank_test_ROC_AUC
8,7,1000,0.864085,0.828697,0.029221,0.035388,0.839603,1
11,10,1000,0.868608,0.828335,0.029145,0.040272,0.843659,2
0,3,250,0.842427,0.828011,0.016053,0.014416,0.826027,3
1,3,500,0.842427,0.828011,0.016053,0.014416,0.826027,3
2,3,1000,0.842427,0.828011,0.016053,0.014416,0.826027,3
5,5,1000,0.857722,0.827661,0.029523,0.030061,0.834224,6


In [69]:
# near best random forest alternatives
rf_near_best = open_rf_results[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__max_features",
        "mean_train_ROC_AUC",
        "mean_test_ROC_AUC",
        "std_test_ROC_AUC",
        "train_val_gap",
        "mean_test_PR_AUC",
        "rank_test_ROC_AUC"
    ]
].sort_values("rank_test_ROC_AUC")

rf_near_best.head(8)

,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features,mean_train_ROC_AUC,mean_test_ROC_AUC,std_test_ROC_AUC,train_val_gap,mean_test_PR_AUC,rank_test_ROC_AUC
14,30,250,sqrt,0.874846,0.837243,0.023368,0.037603,0.859746,1
12,30,50,sqrt,0.885942,0.837104,0.027301,0.048838,0.860623,2
6,20,50,sqrt,0.884992,0.837094,0.026881,0.047899,0.859637,3
13,30,100,sqrt,0.880544,0.837052,0.025495,0.043492,0.859657,4
2,10,250,sqrt,0.869760,0.836972,0.022037,0.032788,0.860264,5
1,10,100,sqrt,0.871395,0.836935,0.023147,0.034460,0.860474,6
8,20,250,sqrt,0.874815,0.836824,0.023456,0.037991,0.859421,7
7,20,100,sqrt,0.880229,0.836676,0.025278,0.043554,0.859313,8


In [70]:
open_carry_forward_params = {
    "logistic_regression__C_expanded": {
        "model__C": 0.001,
        "model__penalty": "l2",
        "model__class_weight": None,
    },

    "decision_tree__C_expanded": {
        "model__max_depth": 3,
        "model__min_samples_leaf": 250,
    },

    "random_forest__C_expanded": {
        "model__n_estimators": 200,
        "model__max_depth": 10,
        "model__min_samples_leaf": 250,
        "model__max_features": "sqrt",
        "model__class_weight": None,
    },
}

In [71]:
open_carry_forward_path = OUTPUT_DIR / "7_open_carry_forward_params_v1.json"

with open(open_carry_forward_path, "w") as f:
    json.dump(open_carry_forward_params, f, indent=4, default=str)

print("Saved open carry-forward params to:")
print(open_carry_forward_path)

Saved open carry-forward params to:
/Users/khanhhien/Documents/Thesis/Outputs/7_open_carry_forward_params_v1.json


## 7.18 Stage 6 vs Stage 7 comparison

This section compares the shortlisted candidate models before and after hyperparameter tuning. The objective is not to re-rank models, but to evaluate whether hyperparameter optimisation improved validation performance and increased cross-validation stability.

Three aspects are considered:

* Validation ROC-AUC (predictive performance)
* Validation standard deviation across folds (stability)

This comparison helps quantify the contribution of Stage 7 beyond simple model selection.

In [73]:
stage6_open_comparison = pd.read_csv(
    OUTPUT_DIR / "6_open_comparison_v1.csv"
)

stage6_shortlisted = stage6_open_comparison[
    stage6_open_comparison["candidate_key"].isin([
        "logistic_regression__C_expanded",
        "decision_tree__C_expanded",
        "random_forest__C_expanded"
    ])
].copy()

stage6_shortlisted = stage6_shortlisted[
    [
        "candidate_key",
        "mean_ROC_AUC",
        "std_ROC_AUC",
        "mean_PR_AUC",
        "mean_brier",
        "mean_log_loss"
    ]
].rename(columns={
    "candidate_key": "candidate",
    "mean_ROC_AUC": "stage6_ROC_AUC",
    "std_ROC_AUC": "stage6_std_ROC_AUC",
    "mean_PR_AUC": "stage6_PR_AUC",
    "mean_brier": "stage6_brier",
    "mean_log_loss": "stage6_log_loss"
})

stage7_summary = open_tuned_summary_df[
    [
        "candidate",
        "validation_score",
        "validation_std",
        "train_val_gap"
    ]
].rename(columns={
    "validation_score": "stage7_ROC_AUC",
    "validation_std": "stage7_std_ROC_AUC",
    "train_val_gap": "stage7_train_val_gap"
})

stage6_vs_stage7 = stage6_shortlisted.merge(
    stage7_summary,
    on="candidate",
    how="left"
)

stage6_vs_stage7["ROC_AUC_change"] = (
    stage6_vs_stage7["stage7_ROC_AUC"]
    - stage6_vs_stage7["stage6_ROC_AUC"]
)

stage6_vs_stage7["std_change"] = (
    stage6_vs_stage7["stage7_std_ROC_AUC"]
    - stage6_vs_stage7["stage6_std_ROC_AUC"]
)

stage6_vs_stage7.sort_values(
    "stage7_ROC_AUC",
    ascending=False
)

,candidate,stage6_ROC_AUC,stage6_std_ROC_AUC,stage6_PR_AUC,stage6_brier,stage6_log_loss,stage7_ROC_AUC,stage7_std_ROC_AUC,stage7_train_val_gap,ROC_AUC_change,std_change
0,random_forest__C_expanded,0.836392,0.026861,0.859215,0.163052,0.496900,0.837243,0.023368,0.037603,0.000851,-0.003493
2,logistic_regression__C_expanded,0.821929,0.026887,0.836937,0.170663,0.519516,0.830110,0.014063,0.021848,0.008181,-0.012823
1,decision_tree__C_expanded,0.826182,0.036542,0.832031,0.166246,0.513710,0.828697,0.029221,0.035388,0.002516,-0.007321


Hyperparameter tuning improved validation ROC-AUC for all shortlisted open-prediction candidates. The largest improvement was observed for Logistic Regression C, where stronger regularisation increased validation ROC-AUC by approximately 0.008 while substantially reducing fold-to-fold variability. Decision Tree C achieved a smaller improvement in validation ROC-AUC, with the main benefit being a clearer understanding of the relationship between model complexity and generalisation. Random Forest C showed only a marginal increase in validation ROC-AUC, suggesting that the configuration identified during broad screening was already close to optimal.

Across all three candidates, validation standard deviation decreased after tuning, indicating improved stability across campaign folds. These results suggest that Stage 7 contributed not only to performance optimisation but also to improved robustness and generalisation.

In [76]:
stage6_vs_stage7_path = (
    OUTPUT_DIR /
    "7_open_stage6_vs_stage7_comparison_v1.csv"
)

stage6_vs_stage7.to_csv(
    stage6_vs_stage7_path,
    index=False
)

print(stage6_vs_stage7_path)

/Users/khanhhien/Documents/Thesis/Outputs/7_open_stage6_vs_stage7_comparison_v1.csv


## 7.19 Decision notes

Hyperparameter tuning was conducted using the train-validation pool only, with grouped stratified cross-validation by mailing_id. The final chronological test set was not used. Although GridSearchCV selected the configuration with the highest mean validation ROC-AUC for each candidate, the final carry-forward decision also considered fold variability, train-validation gap, probability metrics, and model simplicity.

### Logistic Regression C

For Logistic Regression C, the selected configuration was C = 0.001 with L2 regularisation and no class weighting. This setting achieved the highest validation ROC-AUC among the tested logistic configurations and also had the smallest train-validation gap and lowest fold variability. As C increased, validation ROC-AUC declined while the train-validation gap increased, suggesting that weaker regularisation allowed the model to fit patterns that transferred less well across campaign folds. Therefore, the strongest regularisation setting was retained.

### Decision Tree C

For Decision Tree C, the automatic GridSearchCV winner was max_depth = 7 and min_samples_leaf = 1000. However, the simpler depth-3 tree achieved an almost identical validation ROC-AUC, with a substantially smaller train-validation gap and lower fold variability. Since the decision tree is included mainly as a simple nonlinear and interpretable benchmark rather than as the most flexible predictive model, the depth-3 configuration was preferred for carry-forward. At depth 3, the tested min_samples_leaf values produced identical results, indicating that the leaf-size constraint did not materially affect the learned tree at this depth; the first equivalent setting, min_samples_leaf = 250, was retained.

### Random Forest C

For Random Forest C, the automatic GridSearchCV winner was max_depth = 30, min_samples_leaf = 250, max_features = "sqrt", and n_estimators = 200. However, a shallower alternative with max_depth = 10, min_samples_leaf = 250, max_features = "sqrt", and n_estimators = 200 achieved nearly identical validation ROC-AUC while producing a smaller train-validation gap and slightly lower fold variability. Because the performance difference was negligible, the shallower configuration was preferred as the carry-forward random forest model to reduce overfitting risk while preserving predictive performance.

### Conclusion

Overall, the open-prediction tuning results suggest that stronger regularisation or complexity control improved generalisation across campaign folds. The selected configurations therefore prioritise validation performance while avoiding unnecessary model complexity.

## 7.20 Final open tuning conclusion

Open hyperparameter tuning is completed. The final chronological test set was not used during this notebook.

The tuned open candidates carried forward are:

- Logistic Regression C with `C = 0.001`
- Decision Tree C with `max_depth = 3` and `min_samples_leaf = 250`
- Random Forest C with `n_estimators = 200`, `max_depth = 10`, `min_samples_leaf = 250`, and `max_features = "sqrt"`

Although GridSearchCV selected slightly more complex configurations for the decision tree and random forest based on mean validation ROC-AUC, simpler alternatives were preferred because they achieved nearly identical validation ROC-AUC with smaller train-validation gaps and greater stability. This follows the Stage 7 selection rule of preferring simpler or more stable configurations when performance differences are negligible.

The open tuning results show that hyperparameter tuning improved validation ROC-AUC and reduced fold variability compared with Stage 6 broad screening, especially for Logistic Regression C.

In [75]:
open_tuned_summary_path = (
    OUTPUT_DIR /
    "7_open_tuned_model_summary_v1.csv"
)

open_tuned_summary_df.to_csv(
    open_tuned_summary_path,
    index=False
)

print(open_tuned_summary_path)

/Users/khanhhien/Documents/Thesis/Outputs/7_open_tuned_model_summary_v1.csv
